# Muaalem w2v2-384 training on Kaggle (2x T4)

**Before running:**
1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**, Persistence: **Files only**
2. Add-ons → Secrets: add and attach `WANDB_API_KEY`, `HUGGINGFACE_TOKEN` (and `GITHUB_TOKEN` if the repo is private)
3. Fill in the Google Drive links in the **Config** cell (each file shared as "Anyone with the link")
4. For a full run use **Save Version → Save & Run All (Commit)** (runs in the background up to 12h)

## Config

In [ ]:
REPO_URL = "https://github.com/Abdullah-Haytham/prepare-quran-dataset.git"
BRANCH = "saudicomp"
PRIVATE_REPO = False  # True -> uses the GITHUB_TOKEN secret

TRAIN_CONFIG = "configs/train/offline/train_config_w2v2_384_kaggle.yml"

# One Google Drive share link per file: {target_dir: {filename: link}}
# Leave empty ({}) to use the links already written in TRAIN_CONFIG
GDRIVE_FILES = {
    "./w2v2-384": {
        "config.json": "https://drive.google.com/file/d/11L8pAC3djjzdzOhjeT9yJZRY4LQC7XAQ/view?usp=drive_link",
        "model.safetensors": "https://drive.google.com/file/d/1ExzgcmLpfB41Hq2nIxHWvluHDniWkccD/view?usp=drive_link",
        "preprocessor_config.json": "https://drive.google.com/file/d/1p36WQY5zf8GVXHjqm7qM-C9SMnQtnvb3/view?usp=sharing",
    },
}

PUSH_TO_HUB = True
# Hub repo to push to; your HF_TOKEN needs write access to it.
# None -> "<your-hf-username>/muaalem-model-w2v2-384"
HUB_MODEL_ID = None

# To resume after a 12h timeout: add the previous notebook version's output as an
# input and put the path of its results folder here, e.g.
# "/kaggle/input/<notebook-name>/results-w2v2-384"
RESUME_FROM = None

## Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for k in ["WANDB_API_KEY", "HF_TOKEN"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        print(f"Secret `{k}` not found (attach it in Add-ons -> Secrets)")
# train.py's load_secrets() reads HUGGINGFACE_TOKEN
if "HF_TOKEN" in os.environ:
    os.environ["HUGGINGFACE_TOKEN"] = os.environ["HF_TOKEN"]

## Check GPUs

In [ ]:
!nvidia-smi
import torch
print("GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Set Accelerator to 'GPU T4 x2'"

## Clone the repo

In [ ]:
REPO_DIR = "/kaggle/working/prepare-quran-dataset"
url = REPO_URL
if PRIVATE_REPO:
    url = REPO_URL.replace("https://", f"https://{secrets.get_secret('GITHUB_TOKEN')}@")

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD
else:
    !git clone --depth 1 -b {BRANCH} {url} {REPO_DIR}
%cd {REPO_DIR}

## Write Drive links and hub repo into the run config

In [ ]:
import yaml
from huggingface_hub import HfApi

with open(TRAIN_CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if GDRIVE_FILES:
    cfg["gdrive_files"] = GDRIVE_FILES
for d, files in (cfg.get("gdrive_files") or {}).items():
    for name, link in files.items():
        assert "<ID>" not in link, f"Fill in the Google Drive link for {d}/{name}"

if HUB_MODEL_ID is None:
    user = HfApi(token=os.environ.get("HF_TOKEN")).whoami()["name"]
    HUB_MODEL_ID = f"{user}/{cfg['hub_model_id'].split('/')[-1]}"
cfg["hub_model_id"] = HUB_MODEL_ID

TRAIN_CONFIG = "configs/train/offline/_kaggle_run_config.yml"
with open(TRAIN_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print("Using config:", TRAIN_CONFIG)
print("Hub repo:", cfg["hub_model_id"])
print("Output dir:", cfg["output_dir"])

## Restore checkpoints (optional, for resuming)

In [ ]:
import shutil

if RESUME_FROM:
    out = cfg["output_dir"]
    if not os.path.exists(out):
        shutil.copytree(RESUME_FROM, out)
        print(f"Copied {RESUME_FROM} -> {out}")
    !ls {out}

## Install dependencies and download model files from Drive

In [ ]:
import json, subprocess, sys

def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)  # stop the notebook on failure

run("grep qdat kaggle/requirements.txt")  # must show the git+https line; if not, push the latest code
run(f"{sys.executable} -m pip install -q -r kaggle/requirements.txt")
run(f"PYTHONPATH=src:. {sys.executable} -c 'import transformers; print(\"transformers\", transformers.__version__); "
    "import qdat_bench, quran_transcript, multi_level_ctc_model.modeling_multi_level_ctc, "
    "prepare_quran_dataset.modeling_w2v.modeling_multi_level_ctc_w2v; print(\"deps OK\")'")
run(f"{sys.executable} gdrive_download.py --config {TRAIN_CONFIG}")
run("ls -la w2v2-384")
# w2v2bert expects a wav2vec2-bert checkpoint + SeamlessM4TFeatureExtractor
print("model_type:", json.load(open("w2v2-384/config.json")).get("model_type"))
print("feature_extractor_type:", json.load(open("w2v2-384/preprocessor_config.json")).get("feature_extractor_type"))

## Train (DDP on both T4s)

In [ ]:
extra = "--push-to-hub" if PUSH_TO_HUB else ""
run(f"bash kaggle/run_kaggle.sh {TRAIN_CONFIG} {extra}")

## Results

In [ ]:
out = cfg["output_dir"]
if not os.path.exists(out):
    print(f"{out} does not exist: training did not run, check the cells above")
else:
    print(os.listdir(out))
    for name in ["test_results.json", "qdat_bench_test_results.json"]:
        p = os.path.join(out, name)
        if os.path.exists(p):
            print(name); print(open(p).read())